In [1]:
import numpy as np
import pandas as pd

In [2]:
meta_activity_map_file = "/data7/deepro/starrseq/papers/results/2_categorize_fragments_on_activity/data/meta_activity_map.csv"

In [3]:
df = pd.read_csv(meta_activity_map_file, index_col="chrom_coord")
df =  df.fillna(df.median())

In [4]:
libs = ["CC", "ATF2", "CTCF", "FOXA1", "LEF1", "SCRT1", "TCF7L2", "16P12_1"]
stats = ["rpm", "peak", "category"]
diff_stats = ["baseMean", "log2FoldChange", "lfcSE", "stat", "pvalue", "padj", "category"]

# Always active category

In [5]:
libs_cols = [f"{lib}_peak" for lib in libs]
mask = df[libs_cols].eq(1).all(axis=1)

df["CC_category"] = np.where(mask, "always_active", "")

In [6]:
df.CC_category.value_counts()

                 253178
always_active       454
Name: CC_category, dtype: int64

# Always inactive category

In [7]:
libs_cols = [f"{lib}" for lib in libs]
mask = df[libs_cols].lt(-1).all(axis=1)

df.loc[mask, "CC_category"] = "always_inactive"

In [8]:
df.CC_category.value_counts()

                   241909
always_inactive     11269
always_active         454
Name: CC_category, dtype: int64

# Library wise categories

In [9]:
for lib in libs[1:]:
    padj = df[f"{lib}_padj"]
    lfc  = df[f"{lib}_log2FoldChange"]
    peak = df[f"{lib}_peak"] == 1

    cc0 = df["CC_peak"] == 0
    cc1 = df["CC_peak"] == 1
    sig = padj < 0.01
    up  = lfc > 0
    dn  = lfc < 0

    # Define non-overlapping masks (priority order matters)
    gained    = cc0 & sig & peak & up                      # absent in CC, appears/signals in lib
    lost      = cc1 & sig & (~peak) & dn                   # present in CC, disappears/down in lib
    induced   = sig & up  & ~gained                        # significant up but not "gained"
    repressed = sig & dn  & ~lost                          # significant down but not "lost"

    # Anything else is unresponsive for THIS lib
    conds = [gained, lost, induced, repressed]
    choices = ["gained", "lost", "induced", "repressed"]

    df[f"{lib}_category"] = np.select(conds, choices, default="unresponsive")


In [10]:
df.columns = pd.MultiIndex.from_tuples([(c.rsplit("_", 1)[0], c.rsplit("_", 1)[1]) if (len(c.split("_"))>1)&(c.split("_")[-1]!='1') else (c, "rpm") for c in df.columns])

In [11]:
savefile = "/data7/deepro/starrseq/papers/results/2_categorize_fragments_on_activity/data/tables/supplementary_data1.xlsx"

In [12]:
df.loc[:, [("CC", s) for s in stats] + [(l,s) for l in libs[1:] for s in stats[:-1]+diff_stats]].to_excel(savefile)